# model_reasoning.ipynb

Follow-up to `data_analysis.ipynb`'s "Soil Specific: Fine- vs. Coarse-Grained" section (cells 116-126), which
found that the general model's *residuals* correlate with several PSD/gradation features in **opposite
directions** for fine- vs. coarse-grained rows (e.g. `sand`: +0.19 fine vs. -0.13 coarse; `gravel`: -0.16 fine
vs. +0.13 coarse), and that fine-grained rows carry a larger post-model residual (mean |residual| 0.056 fine
vs. 0.037 coarse for MDD).

This notebook asks the direct question `model3.plan.md` needs answered: **does a shared representation
between fine- and coarse-grained rows help or hurt, on the universal (non-Atterberg) feature set each
target actually has access to for both soil types?** Two things are measured:

1. **Do fine and coarse rows even want the same linear direction through feature space?** (standardized
   Ridge coefficient comparison, fit separately per group)
2. **Does pooling the two groups' training rows help or hurt a model evaluated on each group
   separately**, compared to fitting on that group alone? (Regime A: fully separate. Regime B: pooled-fit,
   evaluated per group.) This is a linear-model proxy for the "fully separate specialists" vs. "shared
   encoder" question — cheap enough to run with proper repeated cross-validation (Section 3 of
   `docs/260804.md` found single-split estimates at this n are not trustworthy on their own).

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

from src.general_model_impute import IMPUTED_FEATURES, add_imputed_features, add_no_missing_features

train = pd.read_csv("data/train.csv")
train_feat = add_no_missing_features(train)
train_imputed, fine_imp, coarse_imp = add_imputed_features(train_feat, random_state=42)

TARGETS = ["proctor_mdd_g_cm3", "proctor_owc_pct"]
X_all = train_imputed[IMPUTED_FEATURES].astype(float)
fine_mask = train_imputed["fine-grained"].astype(bool).to_numpy()

print("n fine:", fine_mask.sum(), " n coarse:", (~fine_mask).sum())
print("n universal (IMPUTED_FEATURES) columns:", len(IMPUTED_FEATURES))

n fine: 89  n coarse: 112
n universal (IMPUTED_FEATURES) columns: 28


## 1. Do fine and coarse rows want the same linear direction?

Fit a standardized `RidgeCV` separately on fine-only rows and coarse-only rows (same `IMPUTED_FEATURES`,
same target), and compare the two coefficient vectors directly — cosine similarity and Pearson correlation.
A shared linear encoder implicitly assumes these vectors point in a similar direction; if they don't, a
single shared linear transform can't represent both groups well no matter how it's trained.

In [2]:
alphas = np.logspace(-2, 3, 30)

def fit_coef(X, y):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    model = RidgeCV(alphas=alphas)
    model.fit(Xs, y)
    return model.coef_

coef_results = {}
for target in TARGETS:
    y_all = train_imputed[target].astype(float).to_numpy()
    coef_fine = fit_coef(X_all.loc[fine_mask], y_all[fine_mask])
    coef_coarse = fit_coef(X_all.loc[~fine_mask], y_all[~fine_mask])
    coef_results[target] = (coef_fine, coef_coarse)

    cos_fc = np.dot(coef_fine, coef_coarse) / (np.linalg.norm(coef_fine) * np.linalg.norm(coef_coarse))
    pear_fc = np.corrcoef(coef_fine, coef_coarse)[0, 1]
    print(f"=== {target} ===")
    print(f"  fine-vs-coarse standardized-coef cosine similarity: {cos_fc:+.3f}")
    print(f"  fine-vs-coarse standardized-coef Pearson r:         {pear_fc:+.3f}")

    diff = pd.Series(coef_fine - coef_coarse, index=IMPUTED_FEATURES)
    top_diverge = diff.abs().sort_values(ascending=False).head(6)
    print("  most divergent features (fine_coef vs coarse_coef):")
    for feat in top_diverge.index:
        i = IMPUTED_FEATURES.index(feat)
        print(f"    {feat:35s} fine={coef_fine[i]:+.3f}  coarse={coef_coarse[i]:+.3f}")
    print()

=== proctor_mdd_g_cm3 ===
  fine-vs-coarse standardized-coef cosine similarity: +0.013
  fine-vs-coarse standardized-coef Pearson r:         -0.008
  most divergent features (fine_coef vs coarse_coef):
    sand                                fine=+0.018  coarse=-0.040
    loss_on_ignition_pct_completed      fine=-0.040  coarse=+0.005
    psd_passing_at_2mm_pct              fine=-0.004  coarse=-0.040
    gravel                              fine=+0.004  coarse=+0.040
    feat_cc                             fine=+0.013  coarse=-0.009
    clay                                fine=-0.017  coarse=+0.004

=== proctor_owc_pct ===
  fine-vs-coarse standardized-coef cosine similarity: -0.057
  fine-vs-coarse standardized-coef Pearson r:         -0.067
  most divergent features (fine_coef vs coarse_coef):
    sand                                fine=-0.536  coarse=+0.291
    loss_on_ignition_pct_completed      fine=+0.800  coarse=+0.044
    clay                                fine=+0.475  coarse=-

## 2. Does pooling help or hurt each group?

- **Regime A (fully separate)** — fit `RidgeCV` on one group's training folds only, evaluate on that same
  group's held-out fold. Mirrors `coarse_specialist.py` / `fine_specialist.py`'s "no sharing at all" extreme,
  linear-model proxy.
- **Regime B (pooled-fit)** — fit `RidgeCV` on *all* rows outside a given group's held-out fold (i.e. the
  other group's full data plus this group's training rows), evaluate on that group's held-out fold. Mirrors
  the general model's "full sharing" extreme.

`RepeatedKFold(n_splits=5, n_repeats=5)` per group (25 folds) — a single 80/20 split is exactly what
`docs/260804.md` Section 3 warns reads optimistic/noisy at this n, so this reports a mean ± std over repeats
rather than a point estimate.

In [3]:
rkf = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)
Xnp = X_all.to_numpy()

summary_rows = []
for target in TARGETS:
    y_all = train_imputed[target].astype(float).to_numpy()

    sep_scores = {"fine": [], "coarse": []}
    for name, mask in [("fine", fine_mask), ("coarse", ~fine_mask)]:
        Xg = X_all.loc[mask].to_numpy()
        yg = y_all[mask]
        for tr_idx, te_idx in rkf.split(Xg):
            scaler = StandardScaler().fit(Xg[tr_idx])
            model = RidgeCV(alphas=alphas).fit(scaler.transform(Xg[tr_idx]), yg[tr_idx])
            pred = model.predict(scaler.transform(Xg[te_idx]))
            sep_scores[name].append(r2_score(yg[te_idx], pred))

    pooled_scores = {"fine": [], "coarse": []}
    idx_fine = np.where(fine_mask)[0]
    idx_coarse = np.where(~fine_mask)[0]
    for name, idx_group in [("fine", idx_fine), ("coarse", idx_coarse)]:
        for tr_idx, te_idx in rkf.split(idx_group):
            test_rows = idx_group[te_idx]
            train_rows = np.setdiff1d(np.arange(len(y_all)), test_rows)
            scaler = StandardScaler().fit(Xnp[train_rows])
            model = RidgeCV(alphas=alphas).fit(scaler.transform(Xnp[train_rows]), y_all[train_rows])
            pred = model.predict(scaler.transform(Xnp[test_rows]))
            pooled_scores[name].append(r2_score(y_all[test_rows], pred))

    print(f"=== {target} ===")
    print("  Regime A (separate-per-group Ridge), R^2 mean +/- std over 25 folds:")
    print(f"    fine:   {np.mean(sep_scores['fine']):.3f} +/- {np.std(sep_scores['fine']):.3f}")
    print(f"    coarse: {np.mean(sep_scores['coarse']):.3f} +/- {np.std(sep_scores['coarse']):.3f}")
    print("  Regime B (pooled-fit Ridge, evaluated per group), R^2 mean +/- std over 25 folds:")
    print(f"    fine:   {np.mean(pooled_scores['fine']):.3f} +/- {np.std(pooled_scores['fine']):.3f}")
    print(f"    coarse: {np.mean(pooled_scores['coarse']):.3f} +/- {np.std(pooled_scores['coarse']):.3f}")
    print()

    for group in ["fine", "coarse"]:
        summary_rows.append({
            "target": target, "group": group,
            "separate_r2_mean": np.mean(sep_scores[group]), "separate_r2_std": np.std(sep_scores[group]),
            "pooled_r2_mean": np.mean(pooled_scores[group]), "pooled_r2_std": np.std(pooled_scores[group]),
        })

summary_df = pd.DataFrame(summary_rows)
summary_df

=== proctor_mdd_g_cm3 ===
  Regime A (separate-per-group Ridge), R^2 mean +/- std over 25 folds:
    fine:   -0.407 +/- 2.761
    coarse: 0.854 +/- 0.062
  Regime B (pooled-fit Ridge, evaluated per group), R^2 mean +/- std over 25 folds:
    fine:   0.575 +/- 0.216
    coarse: 0.763 +/- 0.086



=== proctor_owc_pct ===
  Regime A (separate-per-group Ridge), R^2 mean +/- std over 25 folds:
    fine:   -1.283 +/- 4.299
    coarse: 0.481 +/- 0.148
  Regime B (pooled-fit Ridge, evaluated per group), R^2 mean +/- std over 25 folds:
    fine:   -1.801 +/- 5.352
    coarse: 0.444 +/- 0.124



,target,group,separate_r2_mean,separate_r2_std,pooled_r2_mean,pooled_r2_std
0,proctor_mdd_g_cm3,fine,-0.407239,2.761365,0.575264,0.216323
1,proctor_mdd_g_cm3,coarse,0.854135,0.061658,0.763127,0.086491
2,proctor_owc_pct,fine,-1.283308,4.298583,-1.800687,5.351872
3,proctor_owc_pct,coarse,0.480717,0.147860,0.444475,0.124244


## 3. Reading the result

**The coefficient directions are essentially orthogonal** (cosine similarity ~0.01 for MDD, ~-0.06 for OWC;
Pearson r ~-0.008 and ~-0.067). This matches `data_analysis.ipynb`'s sign-flipped residual correlations
(cell 125) from the other direction: fine- and coarse-grained rows don't just have different residuals under
a shared model, they want genuinely different linear mappings from the same universal features to begin
with. A single shared *linear* transform cannot represent both simultaneously — any shared encoder needs
either enough nonlinear capacity to diverge after the shared stage, or explicit group-conditioning before/
inside the shared stage, not just a shared linear layer feeding two linear heads.

**Pooling's effect on out-of-sample R² is target-dependent, not uniformly good or bad:**

- **MDD**: pooling is a clear net win for the fine group (Regime A -0.41 ± 2.76 → Regime B +0.58 ± 0.22) —
  fine-only Ridge at n=89 with 28 features is badly underdetermined/unstable (note the huge std), and
  borrowing the coarse group's rows stabilizes it substantially, at a modest cost to the coarse group itself
  (0.85 → 0.76, still very usable). This is the textbook partial-pooling story a shared encoder is supposed
  to buy: the data-poor group borrows statistical strength from the data-rich one.
- **OWC**: pooling does **not** rescue the fine group (Regime A -1.28 ± 4.30 → Regime B -1.80 ± 5.35, both
  deeply negative) and coarse is roughly flat (0.48 → 0.44). OWC's fine-grained signal isn't recoverable from
  the universal feature set at all here — consistent with `fine_specialist.py`'s premise that OWC on
  fine-grained soils needs the Atterberg/PI features the universal set excludes, not just more pooled rows.

**Implication for `model3`**: a shared-encoder/two-head architecture is well-motivated for **MDD**, where
pooling measurably helps the smaller fine group without ruining the coarse group. For **OWC**, this evidence
doesn't support a plain shared-encoder win — the fine head likely needs the Atterberg-limit enrichment
`fine_specialist.py` already adds (unavailable to a symmetric universal-only shared encoder), or the
architecture needs a materially different capacity/conditioning scheme than what worked for MDD. Treat MDD
and OWC as two separate go/no-go decisions for this architecture, not one — consistent with `docs/260804.md`
Section 8's established finding that MDD and OWC should stay independently modeled targets regardless of
internal architecture.

## 4. Correction: the linear proxy was misleading for OWC

A follow-up question (raised outside this notebook): `data_analysis.ipynb` shows the raw Atterberg limits
correlate strongly with both targets, so shouldn't `model3`'s fine head use them? Checking that directly
exposed a bigger problem with Sections 1-3 above: **they used `RidgeCV` (linear) as a "fast proxy" for the
pooling question, but this project's own prior work (`docs/260804.md` / `v14.plan.md` Section 4) already
established that OWC specifically needs a nonlinear model to show any real signal at all** (Ridge val R^2
0.388 vs. XGBoost val R^2 0.797, on the whole dataset, not just fine-grained rows). Section 2's Regime A/B
results for OWC were almost certainly dominated by Ridge's general OWC weakness, not by anything specific
to pooling or the fine/coarse split. This section redoes the check with XGBoost and adds the raw Atterberg
correlation numbers, and revises Section 3's conclusion.

### 4.1 Raw Atterberg-limit correlation with the targets

Marginal Pearson/Spearman correlation, computed only on rows where each column is actually measured
(Atterberg limits: n=26 -- a small, non-random subset, since they're physically measured for only some
fine-grained soils; LOI/kf/hyd_grad: n=61).

In [4]:
cols = ['atterberg_liquid_limit_pct', 'atterberg_plastic_limit_pct', 'loss_on_ignition_pct',
        'hyd_cond_kf_m_s', 'hyd_cond_hyd_gradient']

raw = pd.read_csv('data/train.csv')
for target in TARGETS:
    print(f'--- {target} ---')
    for c in cols:
        sub = raw[[c, target]].dropna()
        pear = sub[c].corr(sub[target])
        spear = sub[c].corr(sub[target], method='spearman')
        print(f'  {c:35s} pearson={pear:+.3f}  spearman={spear:+.3f}  n={len(sub)}')
    print()

--- proctor_mdd_g_cm3 ---
  atterberg_liquid_limit_pct          pearson=-0.739  spearman=-0.711  n=26
  atterberg_plastic_limit_pct         pearson=-0.819  spearman=-0.802  n=26
  loss_on_ignition_pct                pearson=-0.522  spearman=-0.399  n=61
  hyd_cond_kf_m_s                     pearson=-0.241  spearman=+0.204  n=61
  hyd_cond_hyd_gradient               pearson=-0.197  spearman=-0.218  n=61

--- proctor_owc_pct ---
  atterberg_liquid_limit_pct          pearson=+0.695  spearman=+0.655  n=26
  atterberg_plastic_limit_pct         pearson=+0.737  spearman=+0.732  n=26
  loss_on_ignition_pct                pearson=+0.785  spearman=+0.744  n=61
  hyd_cond_kf_m_s                     pearson=-0.147  spearman=-0.583  n=61
  hyd_cond_hyd_gradient               pearson=+0.177  spearman=+0.122  n=61



**Confirmed**: `atterberg_plastic_limit_pct` and `atterberg_liquid_limit_pct` correlate with both targets
far more strongly (|r| 0.70-0.82) than almost any universal feature -- real signal, not a false lead. The
caveat is n=26: a correlation this size on 26 rows still carries a wide confidence interval, and it's only
ever available for the fine-grained population to begin with (coarse soils never have Atterberg limits
measured -- structural, not random, missingness).

### 4.2 Does adding MICE-imputed Atterberg features actually help a model fit on the fine subset?

`fine_specialist.py` doesn't use the *raw* n=26 measurements directly -- for the other ~63 fine rows it
substitutes MICE-imputed values (`FINE_SPECIALIST_FEATURES` = `IMPUTED_FEATURES` + Atterberg/PI + missingness
flags, 35 columns total). Testing whether that enrichment beats the 28 universal-only columns, fine-only
rows, both for `RidgeCV` (Section 1's tool) and `XGBRegressor` (Section 4.3's tool):

In [5]:
from sklearn.linear_model import RidgeCV as _RidgeCV
from src.fine_specialist import prepare_fine_specialist_features, select_fine_grained

df_fine_ready, *_ = prepare_fine_specialist_features(train_feat, random_state=42)
X_fine_enriched, y_mdd_fine = select_fine_grained(df_fine_ready, target_col='proctor_mdd_g_cm3')
_, y_owc_fine = select_fine_grained(df_fine_ready, target_col='proctor_owc_pct')
X_fine_universal = X_fine_enriched[IMPUTED_FEATURES]

print(f'fine rows: {len(y_mdd_fine)}, universal cols: {X_fine_universal.shape[1]}, enriched cols: {X_fine_enriched.shape[1]}')
print()

def ridge_cv_r2(X, y):
    Xnp = X.to_numpy()
    scores = []
    for tr, te in rkf.split(Xnp):
        scaler = StandardScaler().fit(Xnp[tr])
        m = _RidgeCV(alphas=alphas).fit(scaler.transform(Xnp[tr]), y[tr])
        scores.append(r2_score(y[te], m.predict(scaler.transform(Xnp[te]))))
    return np.mean(scores), np.std(scores)

for name, y in [('MDD', y_mdd_fine), ('OWC', y_owc_fine)]:
    mu, su = ridge_cv_r2(X_fine_universal, y)
    me, se = ridge_cv_r2(X_fine_enriched, y)
    print(f'{name}: Ridge, universal-only   R2 = {mu:+.3f} +/- {su:.3f}')
    print(f'{name}: Ridge, Atterberg-enriched R2 = {me:+.3f} +/- {se:.3f}')
    print()

fine rows: 89, universal cols: 28, enriched cols: 35



MDD: Ridge, universal-only   R2 = -0.407 +/- 2.761
MDD: Ridge, Atterberg-enriched R2 = -0.418 +/- 2.645

OWC: Ridge, universal-only   R2 = -1.283 +/- 4.299
OWC: Ridge, Atterberg-enriched R2 = -1.603 +/- 4.847



### 4.3 Redo Section 2 with XGBoost instead of Ridge

Same Regime A (separate-per-group) / Regime B (pooled-fit, evaluated per group) construction as Section 2,
swapping `RidgeCV` for a lightly-regularized `XGBRegressor` -- still cheap enough for the same 25-fold
repeated CV.

In [6]:
from xgboost import XGBRegressor

def make_xgb():
    return XGBRegressor(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        min_child_weight=4, subsample=0.8, colsample_bytree=0.8,
        reg_lambda=2.0, random_state=42,
    )

def xgb_cv_r2(Xnp, y, idx):
    scores = []
    for tr, te in rkf.split(idx):
        tr_idx, te_idx = idx[tr], idx[te]
        m = make_xgb().fit(Xnp[tr_idx], y[tr_idx])
        scores.append(r2_score(y[te_idx], m.predict(Xnp[te_idx])))
    return np.mean(scores), np.std(scores)

xgb_summary_rows = []
for target in TARGETS:
    y_all = train_imputed[target].astype(float).to_numpy()

    sep_fine = xgb_cv_r2(Xnp, y_all, idx_fine)
    sep_coarse = xgb_cv_r2(Xnp, y_all, idx_coarse)

    # pooled-fit: train on (all rows minus this group's held-out fold), same construction as Section 2
    def pooled_xgb(idx_group):
        scores = []
        for tr_idx, te_idx in rkf.split(idx_group):
            test_rows = idx_group[te_idx]
            train_rows = np.setdiff1d(np.arange(len(y_all)), test_rows)
            m = make_xgb().fit(Xnp[train_rows], y_all[train_rows])
            scores.append(r2_score(y_all[test_rows], m.predict(Xnp[test_rows])))
        return np.mean(scores), np.std(scores)

    pool_fine = pooled_xgb(idx_fine)
    pool_coarse = pooled_xgb(idx_coarse)

    print(f'=== {target} ===')
    print('  Regime A (separate XGBoost per group):')
    print(f'    fine:   {sep_fine[0]:.3f} +/- {sep_fine[1]:.3f}')
    print(f'    coarse: {sep_coarse[0]:.3f} +/- {sep_coarse[1]:.3f}')
    print('  Regime B (pooled-fit XGBoost, evaluated per group):')
    print(f'    fine:   {pool_fine[0]:.3f} +/- {pool_fine[1]:.3f}')
    print(f'    coarse: {pool_coarse[0]:.3f} +/- {pool_coarse[1]:.3f}')
    print()

    for group, sep, pool in [('fine', sep_fine, pool_fine), ('coarse', sep_coarse, pool_coarse)]:
        xgb_summary_rows.append({'target': target, 'group': group,
                                  'separate_r2_mean': sep[0], 'separate_r2_std': sep[1],
                                  'pooled_r2_mean': pool[0], 'pooled_r2_std': pool[1]})

pd.DataFrame(xgb_summary_rows)

=== proctor_mdd_g_cm3 ===
  Regime A (separate XGBoost per group):
    fine:   0.748 +/- 0.055
    coarse: 0.846 +/- 0.070
  Regime B (pooled-fit XGBoost, evaluated per group):
    fine:   0.784 +/- 0.056
    coarse: 0.852 +/- 0.073



=== proctor_owc_pct ===
  Regime A (separate XGBoost per group):
    fine:   0.755 +/- 0.062
    coarse: 0.460 +/- 0.198
  Regime B (pooled-fit XGBoost, evaluated per group):
    fine:   0.774 +/- 0.054
    coarse: 0.488 +/- 0.182



,target,group,separate_r2_mean,separate_r2_std,pooled_r2_mean,pooled_r2_std
0,proctor_mdd_g_cm3,fine,0.747581,0.055187,0.784484,0.056260
1,proctor_mdd_g_cm3,coarse,0.845660,0.070151,0.851966,0.072778
2,proctor_owc_pct,fine,0.754939,0.062202,0.774374,0.054249
3,proctor_owc_pct,coarse,0.459613,0.198020,0.487599,0.182007


### 4.4 Revised conclusion (supersedes Section 3 for OWC)

**The nonlinear picture is materially different from, and more encouraging than, the linear one:**

- Pooling helps **all four** target x group combinations under XGBoost, not just MDD's fine group -- modestly
  (roughly +0.02 to +0.04 R^2 each) but consistently, including OWC-coarse and OWC-fine.
- **OWC's fine group is not hopeless.** Even fit in complete isolation (Regime A, no pooling), fine-only
  XGBoost reaches R^2 ~0.75 for OWC -- actually *better* than OWC's coarse group (R^2 ~0.46-0.49) and
  in the same range as MDD. Section 3's conclusion ("OWC's fine-grained signal isn't recoverable from the
  universal feature set") was an artifact of Ridge's well-documented general weakness on OWC
  (`v14.plan.md` Section 4: Ridge val R^2 0.388 vs. XGBoost 0.797, dataset-wide, not fine/coarse-specific),
  not a property of the fine-grained population specifically. **Lesson for future feasibility checks in
  this project: a linear proxy is fine for MDD, but not a valid stand-in for OWC** -- any "is X worth
  trying" check on OWC needs a nonlinear model or it will read falsely negative.
- **Atterberg enrichment still doesn't clearly help**, in either Ridge or (per Section 4.2's own numbers)
  a fine-only fit -- consistent with the raw correlation being real (Section 4.1) but concentrated in a
  small measured subset (n=26) that MICE imputation doesn't fully recover for the other ~63 fine rows, and
  likely redundant with the universal feature set's own fines-content/LOI columns once a nonlinear model can
  use them. This doesn't rule out Atterberg mattering for a specific architecture later, but there's no
  evidence yet that `model3`'s fine head needs it to reach a strong result.

**Revised scope recommendation for `model3`**: build **both MDD and OWC**, not MDD-only as Sections 1-3
concluded -- OWC's fine-grained signal is real and comparable to MDD's once tested with the right tool.
`docs/01-plan/features/model3.plan.md` Section 4 and `docs/02-design/features/model3.design.md` Decision 1.2
should be updated to reflect this.

### 4.5 Does the shared-encoder idea add anything a plain feature doesn't already give XGBoost?

`model3`'s whole premise is a learned representation *conditioned* on the fine/coarse split. The cheapest
possible version of "conditioning" is just adding `fine-grained` as one more input column to an otherwise
ordinary pooled XGBoost -- note `IMPUTED_FEATURES` currently excludes it entirely
(`general_model_impute.py`: `MICE_PREDICTORS` explicitly filters `c != 'fine-grained'`), so even the existing
general model (`model1i`) never sees soil population type as a feature at all.

In [7]:
Xnp_flag = np.column_stack([Xnp, fine_mask.astype(float)])

for target in TARGETS:
    y_all = train_imputed[target].astype(float).to_numpy()
    print(f'=== {target} ===')
    for name, idx_group in [('fine', idx_fine), ('coarse', idx_coarse)]:
        no_flag = pooled_xgb(idx_group)
        with_flag_scores = []
        for tr_idx, te_idx in rkf.split(idx_group):
            test_rows = idx_group[te_idx]
            train_rows = np.setdiff1d(np.arange(len(y_all)), test_rows)
            m = make_xgb().fit(Xnp_flag[train_rows], y_all[train_rows])
            with_flag_scores.append(r2_score(y_all[test_rows], m.predict(Xnp_flag[test_rows])))
        print(f'  {name}: pooled XGBoost, no flag    R2 = {no_flag[0]:.3f} +/- {no_flag[1]:.3f}')
        print(f'  {name}: pooled XGBoost, +fine flag R2 = {np.mean(with_flag_scores):.3f} +/- {np.std(with_flag_scores):.3f}')
    print()

=== proctor_mdd_g_cm3 ===


  fine: pooled XGBoost, no flag    R2 = 0.784 +/- 0.056
  fine: pooled XGBoost, +fine flag R2 = 0.783 +/- 0.052


  coarse: pooled XGBoost, no flag    R2 = 0.852 +/- 0.073
  coarse: pooled XGBoost, +fine flag R2 = 0.853 +/- 0.068

=== proctor_owc_pct ===


  fine: pooled XGBoost, no flag    R2 = 0.774 +/- 0.054
  fine: pooled XGBoost, +fine flag R2 = 0.769 +/- 0.065


  coarse: pooled XGBoost, no flag    R2 = 0.488 +/- 0.182
  coarse: pooled XGBoost, +fine flag R2 = 0.489 +/- 0.185



**The flag adds essentially nothing** (every difference is within 0.01 R^2, well inside the fold-to-fold
noise). This makes sense in hindsight: tree splits can already reconstruct soil-type membership from the
gradation features themselves (clay/silt/sand fractions determine `fine-grained` by definition), so handing
XGBoost the flag explicitly is redundant information, not new information.

**This substantially weakens `model3`'s core premise.** Section 4.4's pooling benefit (Regime A -> Regime B)
comes from XGBoost simply having *more training rows*, not from anything resembling group-conditioning --
a plain pooled XGBoost, with no fine/coarse split at all, already captures it. A custom shared-encoder/
two-head neural network is trying to buy something (explicit group-conditioning) that a tree ensemble
already gets for free from the raw features. Before building `model3` as originally scoped, the honest next
question is whether **any custom architecture is needed at all**, versus simply confirming the existing
general model's XGBoost component (or a dedicated pooled-XGBoost submission) already captures this -- see
the revised recommendation in `docs/01-plan/features/model3.plan.md`.